# MuonClip + RMS: layer ESDs and `fix_fingers='clip_xmax'`

This notebook inspects all six one-head nanoGPT hidden matrices at an epoch checkpoint.

**The run directory comes from the shell environment variable `RUN_DIR`.**
Set it in the same shell before launching Jupyter, for example:

```bash
export RUN_DIR=/tmp/rg-nanogpt-long-muonclip-50ep/results/muon_clip/seed_1337
echo "$RUN_DIR"
jupyter lab notebooks/05_muonclip_esd_clip_xmax.ipynb
```

The notebook never hard-codes a run path. It loads the latest completed epoch/quarter-epoch checkpoint by default, runs WeightWatcher exactly once with `fix_fingers='clip_xmax'`, compares `raw_alpha` with the corrected `alpha`, and plots a 2x3 ESD grid.


In [ ]:
from pathlib import Path
import json
import os
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
import weightwatcher as ww

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / "baseline" / "nanogpt_one_head"]
EXPERIMENT_ROOT = next(
    (p for p in candidates if (p / "configs" / "reference.yaml").is_file()),
    None,
)
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError("Run from baseline/nanogpt_one_head or the repository root")

sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
from rg_nanogpt_one_head.model import GPT, GPTConfig
from rg_nanogpt_one_head.spectral import WeightMatrixHolder, _attach_matrix_metadata

run_dir_env = os.environ.get("RUN_DIR", "").strip()
if not run_dir_env:
    raise EnvironmentError(
        "RUN_DIR is not set. In the shell that launches Jupyter, run:\n"
        "  export RUN_DIR=/path/to/results/<optimizer>/seed_<seed>\n"
        "  echo \"$RUN_DIR\""
    )
RUN_DIR = Path(run_dir_env).expanduser().resolve()

# None = latest completed epoch/quarter-epoch checkpoint.
TARGET_EPOCH = None

MIN_EVALS = 20
MAX_FINGERS = 10
RANDOMIZE = True
ERG = True

print("experiment root:", EXPERIMENT_ROOT)
print("RUN_DIR from environment:", RUN_DIR)
print("WeightWatcher version:", getattr(ww, "__version__", "unknown"))

## Load the checkpoint

`epoch_metrics.csv` contains the model-only checkpoint path for each nominal epoch checkpoint. `TARGET_EPOCH=None` selects the latest available row. Set `TARGET_EPOCH = 4.0`, for example, to inspect epoch 4 explicitly.


In [ ]:
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Run directory does not exist: {RUN_DIR}")

manifest = json.loads((RUN_DIR / "manifest.json").read_text())
epoch_metrics = pd.read_csv(RUN_DIR / "epoch_metrics.csv")
for column in ["step", "nominal_epoch", "epoch"]:
    epoch_metrics[column] = pd.to_numeric(epoch_metrics[column], errors="coerce")
epoch_metrics = epoch_metrics.dropna(subset=["step", "nominal_epoch"]).sort_values("step")

if TARGET_EPOCH is None:
    selected = epoch_metrics.iloc[-1]
else:
    idx = (epoch_metrics["nominal_epoch"] - float(TARGET_EPOCH)).abs().idxmin()
    selected = epoch_metrics.loc[idx]

STEP = int(selected["step"])
NOMINAL_EPOCH = float(selected["nominal_epoch"])
ACTUAL_EPOCH = float(selected["epoch"])

checkpoint_path = Path(str(selected.get("checkpoint_path", "")))
if not checkpoint_path.is_file():
    matches = sorted((RUN_DIR / "epoch_checkpoints").glob(f"*step_{STEP:07d}.pt"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Could not resolve checkpoint for step {STEP}; found {matches}"
        )
    checkpoint_path = matches[0]

payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model = GPT(GPTConfig(**manifest["model"]))
model.load_state_dict(payload["model"])
model.eval()

holder = WeightMatrixHolder(model)
OUTPUT_DIR = RUN_DIR / "diagnostics" / f"esd_clip_xmax_step_{STEP:07d}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("checkpoint:", checkpoint_path)
print(f"step={STEP} nominal_epoch={NOMINAL_EPOCH:.3f} actual_epoch={ACTUAL_EPOCH:.6f}")
print("matrices:", [m["matrix_name"] for m in holder.matrix_metadata])
print("output dir:", OUTPUT_DIR)

stored_path = RUN_DIR / "spectral" / "raw" / f"weightwatcher_step_{STEP:07d}.csv"
if stored_path.is_file():
    stored = pd.read_csv(stored_path)
    cols = [c for c in ["matrix_name","alpha","D","rand_distance","ERG_gap","num_traps"] if c in stored.columns]
    print("\nStored training-time WeightWatcher result:")
    display(stored[cols].sort_values("matrix_name"))

## One-pass WeightWatcher analysis

This reproduces the training-time contract and uses the same six-matrix holder. WeightWatcher 0.7.7 returns both the corrected `alpha` and pre-finger-removal `raw_alpha` from this single call.


In [ ]:
diagnostic_seed = int(manifest["seed"]) + 1_000_003 + STEP

def reset_diagnostic_seed():
    random.seed(diagnostic_seed)
    np.random.seed(diagnostic_seed % (2**32 - 1))
    torch.manual_seed(diagnostic_seed)

def attach(details):
    return _attach_matrix_metadata(pd.DataFrame(details), holder.matrix_metadata)

def show_details(frame):
    preferred = [
        "matrix_name", "alpha", "raw_alpha", "D", "xmin", "xmax",
        "num_fingers", "num_pl_spikes", "ERG_gap", "rand_distance", "warning",
    ]
    cols = [c for c in preferred if c in frame.columns]
    display(frame[cols].sort_values("matrix_name"))

reset_diagnostic_seed()
watcher = ww.WeightWatcher(model=holder)
details_raw = watcher.analyze(
    ERG=ERG,
    randomize=RANDOMIZE,
    plot=True,
    min_evals=MIN_EVALS,
    fix_fingers='clip_xmax',
    max_fingers=MAX_FINGERS,
)
details = attach(details_raw)
details["alpha_raw"] = pd.to_numeric(details["raw_alpha"], errors="coerce")
details["alpha_clip_xmax"] = pd.to_numeric(details["alpha"], errors="coerce")
details["alpha_reduction"] = details["alpha_raw"] - details["alpha_clip_xmax"]
details["weightwatcher_analysis_calls"] = 1
details.to_csv(OUTPUT_DIR / "weightwatcher_one_pass_clip_xmax.csv", index=False)
show_details(details)

## `fix_fingers='clip_xmax'`

No second analysis is needed: `raw_alpha` and corrected `alpha` come from the one call above. The separate raw fit boundaries are not claimed because this one-pass contract does not return them.


In [ ]:
required = {"alpha", "raw_alpha", "num_fingers"}
missing = required.difference(details.columns)
if missing:
    raise RuntimeError(f"one-pass WeightWatcher result is missing: {sorted(missing)}")
if not details["weightwatcher_analysis_calls"].eq(1).all():
    raise RuntimeError("WeightWatcher must be called exactly once")
show_details(details)

## Raw versus clipped alpha from the same call


In [ ]:
comparison_columns = [
    "matrix_name", "alpha_raw", "alpha_clip_xmax", "alpha_reduction",
    "D", "xmin", "xmax", "num_fingers", "num_pl_spikes",
]
comparison = details[[c for c in comparison_columns if c in details.columns]].sort_values("matrix_name")
comparison.to_csv(OUTPUT_DIR / "raw_vs_clip_xmax_one_pass.csv", index=False)
display(comparison)

if "alpha_raw" in comparison and "alpha_clip_xmax" in comparison:
    print("median alpha, raw      :", float(comparison["alpha_raw"].median()))
    print("median alpha, clip_xmax:", float(comparison["alpha_clip_xmax"].median()))
    print("mean alpha, raw        :", float(comparison["alpha_raw"].mean()))
    print("mean alpha, clip_xmax  :", float(comparison["alpha_clip_xmax"].mean()))

## All six ESDs

The empirical ESD is unchanged by `clip_xmax`. The vertical lines show the corrected fit's `xmin` and `xmax`; a separate raw-fit boundary is not fabricated.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
axes = axes.ravel()

for ax, (_, row) in zip(
    axes, details.sort_values("matrix_name").iterrows()
):
    matrix_name = str(row["matrix_name"])
    layer_id = int(row["layer_id"])
    esd = np.asarray(watcher.get_ESD(layer=layer_id), dtype=float)
    esd = esd[np.isfinite(esd) & (esd > 0)]

    if esd.size == 0:
        ax.set_title(f"{matrix_name}: no positive eigenvalues")
        continue

    lo, hi = float(esd.min()), float(esd.max())
    bins = (
        np.linspace(lo * 0.99, hi * 1.01 + 1e-12, 16)
        if hi <= lo
        else np.logspace(np.log10(lo), np.log10(hi), 32)
    )
    hist, edges = np.histogram(esd, bins=bins, density=True)
    centers = np.sqrt(edges[:-1] * edges[1:])
    valid = np.isfinite(hist) & (hist > 0) & np.isfinite(centers) & (centers > 0)
    ax.loglog(centers[valid], hist[valid], marker="o", linestyle="none", label="ESD")

    for value, label, style in [
        (row.get("xmin", np.nan), "xmin clip_xmax", ":"),
        (row.get("xmax", np.nan), "xmax clip_xmax", "-."),
    ]:
        value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
        if np.isfinite(value) and value > 0:
            ax.axvline(float(value), linestyle=style, label=label)

    alpha_raw = float(row["raw_alpha"]) if pd.notna(row.get("raw_alpha")) else float("nan")
    alpha_clip = float(row["alpha"]) if pd.notna(row.get("alpha")) else float("nan")
    ax.set_title(f"{matrix_name}  alpha: raw {alpha_raw:.3f} -> clip {alpha_clip:.3f}")
    ax.set_xlabel("eigenvalue of X = W^T W")
    ax.set_ylabel("density")
    ax.grid(True, which="both", alpha=0.2)
    ax.legend(fontsize=8)

fig.suptitle(
    f"MuonClip + RMS, epoch {NOMINAL_EPOCH:.2f}, step {STEP}: ESD + clip_xmax",
    fontsize=14,
)
figure_path = OUTPUT_DIR / "all_layer_esds_raw_vs_clip_xmax_one_pass.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", figure_path)

## Interpretation

- Large `alpha_reduction` with nonzero `num_fingers`: the raw alpha was strongly influenced by top-of-spectrum finite-size fingers.
- Little alpha reduction: the high alpha is not explained by those fingers.
- Always inspect `D` together with alpha.
- Outputs are saved below `$RUN_DIR/diagnostics/esd_clip_xmax_step_XXXXXXX/`.
